# SQLite Database — FinTech 590

Loads pool snapshots and historical data into a local SQLite database.

**Run order:** `defi_pipeline.ipynb` → `historical_data.ipynb` → this notebook.

**Schema:**
```
pools          — one row per pool (current snapshot)
pool_history   — one row per pool per day (historical TVL, APY, IL)
```

## 0. Imports

In [1]:
import sqlite3, pathlib
from datetime import date
import pandas as pd

DB_PATH          = pathlib.Path("data/defi_pools.db")
POOLS_PARQUET    = pathlib.Path("data/top_pools.parquet")
HISTORY_PARQUET  = pathlib.Path("data/pool_history.parquet")

print(f"Database : {DB_PATH}")
print(f"Pools    : {POOLS_PARQUET}")
print(f"History  : {HISTORY_PARQUET}")

Database : data\defi_pools.db
Pools    : data\top_pools.parquet
History  : data\pool_history.parquet


## 1. Create Tables

Uses `CREATE TABLE IF NOT EXISTS` — safe to re-run without wiping data.

In [ ]:
con = sqlite3.connect(DB_PATH)
cur = con.cursor()

cur.execute("""
    CREATE TABLE IF NOT EXISTS pools (
        address            TEXT,
        chain              TEXT,
        token0             TEXT,
        token1             TEXT,
        fee_tier           INTEGER,
        tvl_usd            REAL,
        volume_usd         REAL,
        etherscan_verified INTEGER,
        fetched_at         TEXT,
        PRIMARY KEY (address, chain)
    )
""")

cur.execute("""
    CREATE TABLE IF NOT EXISTS pool_history (
        address     TEXT,
        chain       TEXT,
        date        TEXT,
        tvl_usd     REAL,
        apy         REAL,
        apy_base    REAL,
        apy_base_7d REAL,
        il_7d       REAL,
        PRIMARY KEY (address, chain, date),
        FOREIGN KEY (address, chain) REFERENCES pools(address, chain)
    )
""")
con.commit()
print("Tables created (or already exist).")

# Print schema
for (t,) in cur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall():
    print(f"\n  [{t}]")
    for col in cur.execute(f"PRAGMA table_info({t})").fetchall():
        print(f"    {col[1]:20s} {col[2]}" + (" <- PK" if col[5] else ""))

## 2. Load Pools

Uses `INSERT OR REPLACE` — re-running always reflects the latest snapshot.

In [ ]:
pools_df = pd.read_parquet(POOLS_PARQUET)
pools_df["fetched_at"] = str(date.today())

pools_df[["address", "chain", "token0", "token1", "fee_tier",
           "tvl_usd", "volume_usd",
           "etherscan_verified", "fetched_at"]].to_sql(
    "pools", con, if_exists="replace", index=False
)
con.commit()

count = cur.execute("SELECT COUNT(*) FROM pools").fetchone()[0]
print(f"pools table: {count} rows loaded")
pd.read_sql(
    "SELECT chain, address, token0, token1, fee_tier, tvl_usd, etherscan_verified FROM pools ORDER BY chain, tvl_usd DESC",
    con
)

## 3. Load Historical Data

Uses `INSERT OR IGNORE` — re-running skips existing (address, date) pairs so no duplicates are added.

In [ ]:
history_df = pd.read_parquet(HISTORY_PARQUET)
history_df["date"] = history_df["date"].dt.strftime("%Y-%m-%d")

rows = history_df[["address", "chain", "date", "tvl_usd", "apy",
                    "apy_base", "apy_base_7d", "il_7d"]].values.tolist()

cur.executemany("""
    INSERT OR IGNORE INTO pool_history
        (address, chain, date, tvl_usd, apy, apy_base, apy_base_7d, il_7d)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", rows)
con.commit()

count      = cur.execute("SELECT COUNT(*) FROM pool_history").fetchone()[0]
date_range = cur.execute("SELECT MIN(date), MAX(date) FROM pool_history").fetchone()
chains     = cur.execute("SELECT DISTINCT chain FROM pool_history ORDER BY chain").fetchall()
print(f"pool_history table: {count:,} rows")
print(f"Date range        : {date_range[0]}  ->  {date_range[1]}")
print(f"Chains            : {', '.join(r[0] for r in chains)}")

## 4. Example Queries

In [ ]:
# Top pools by average TVL across all history
print("Top pools by average historical TVL")
pd.read_sql("""
    SELECT p.chain, p.token0, p.token1, p.fee_tier,
           ROUND(AVG(h.tvl_usd) / 1e6, 2) AS avg_tvl_usd_m,
           ROUND(MAX(h.tvl_usd) / 1e6, 2) AS peak_tvl_usd_m,
           COUNT(h.date)                   AS days_tracked
    FROM pool_history h
    JOIN pools p ON h.address = p.address AND h.chain = p.chain
    GROUP BY h.address, h.chain
    ORDER BY avg_tvl_usd_m DESC
""", con)

In [ ]:
# Average base APY by pool — last 90 days
print("Average base APY — last 90 days")
pd.read_sql("""
    SELECT p.chain, p.token0, p.token1, p.fee_tier,
           ROUND(AVG(h.apy_base), 2) AS avg_apy_base_pct,
           ROUND(MAX(h.apy_base), 2) AS max_apy_base_pct
    FROM pool_history h
    JOIN pools p ON h.address = p.address AND h.chain = p.chain
    WHERE h.date >= date('now', '-90 days')
      AND h.apy_base IS NOT NULL
    GROUP BY h.address, h.chain
    ORDER BY avg_apy_base_pct DESC
""", con)

In [ ]:
# Monthly average TVL for top 5 pools
print("Monthly average TVL — top 5 pools")
pd.read_sql("""
    SELECT p.chain, p.token0 || '/' || p.token1 AS pair,
           strftime('%Y-%m', h.date)             AS month,
           ROUND(AVG(h.tvl_usd) / 1e6, 2)       AS avg_tvl_usd_m
    FROM pool_history h
    JOIN pools p ON h.address = p.address AND h.chain = p.chain
    WHERE (p.address, p.chain) IN (
        SELECT address, chain FROM pools ORDER BY tvl_usd DESC LIMIT 5
    )
    GROUP BY p.address, p.chain, month
    ORDER BY month DESC, avg_tvl_usd_m DESC
    LIMIT 30
""", con)

In [8]:
con.close()
print(f"Database saved to {DB_PATH}  ({DB_PATH.stat().st_size / 1024:.0f} KB)")

Database saved to data\defi_pools.db  (3208 KB)
